# Create Feature03.csv

產生金額相關的特徵，依照匯率轉換為台幣  
匯率參考時期: 2024/12 - 2025/03

Output: `feature03.csv`  

Form:
| acct | send_sum_twd | send_count_twd | send_mean_twd | send_std_twd | send_min_twd | send_max_twd | recv_sum_twd | recv_count_twd | recv_mean_twd | recv_std_twd | recv_min_twd | recv_max_twd | 
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |

Descriptions:
| Column | Description | Range |
| --- | --- | --- |
| send_sum_twd | 帳號「匯款」的金額總和 |  float |
| send_count_twd | 帳號「匯款」的次數 | int |
| send_mean_twd | 帳號「匯款」的金額平均 | float |
| send_std_twd | 帳號「匯款」的金額標準差 | float |
| send_min_twd | 帳號「匯款」的金額最小值 | float |
| send_max_twd | 帳號「匯款」的金額最大值 | float |
| --- | --- | --- |
| recv_sum_twd |  帳號「收款」的金額總和 | float |
| recv_count_twd | 帳號「收款」的次數 | int |
| recv_mean_twd | 帳號「收款」的金額平均 | float |
| recv_std_twd | 帳號「收款」的金額最大值 | float |
| recv_min_twd | 帳號「收款」的金額最小值 | float |
| recv_max_twd | 帳號「收款」的金額最大值 | float |

In [1]:
import pandas as pd

### Read files

In [2]:
column_types = {
    "from_acct": "category",
    "from_acct_type": "category",
    "to_acct": "category",
    "to_acct_type": "category",
    "is_self_txn": "category",
    "txn_amt": "float",
    "txn_date": "category",
    "txn_time": "category",
    "currency_type": "category",
    "channel_type": "category"
}
alert_types = {
    "acct": "category",
    "event_date": "category"
}

df_txn = pd.read_csv("../src/acct_transaction.csv", dtype=column_types)
df_alert = pd.read_csv("../src/acct_alert.csv", dtype=alert_types)

print("CSV files read complete.")

CSV files read complete.


### 轉換匯率

In [3]:
exchange_rate = {
        'TWD': 1.0, 'USD': 32.5, 'JPY': 0.22, 'CNY': 4.5, 
        'HKD': 4.2, 'EUR': 34.0, 'GBP': 41.0, 'AUD': 20.5, 
        'CAD': 22.8, 'NZD': 18.5, 'THB': 0.97, 'ZAR': 1.77, 
        'SGD': 24.3, 'CHF': 36.5, 'SEK': 3.1, 'MXN': 1.6
    }

# 確保型態正確
df_txn['txn_amt'] = df_txn['txn_amt'].astype(float)
df_txn['currency_type'] = df_txn['currency_type'].astype(str)

# 將匯率應用到交易金額，轉換為 TWD
df_txn['txn_amt_twd'] = df_txn.apply(
    lambda x: x['txn_amt'] * exchange_rate.get(x['currency_type']),
    axis=1
)

### 處理資料

In [4]:
# get send stats
print("getting send stats...")
send_stats = df_txn.groupby('from_acct').agg(
    send_sum=('txn_amt_twd', 'sum'), 
    send_count=('txn_amt_twd', 'count'), 
    send_std=('txn_amt_twd', 'std'), 
    send_mean=('txn_amt_twd', 'mean'), 
    send_min=('txn_amt_twd', 'min'),
    send_max=('txn_amt_twd', 'max')
).reset_index().rename(columns={'from_acct': 'acct'})

# get recv stats
print("getting recv stats...")
recv_stats = df_txn.groupby('to_acct').agg(
    recv_sum=('txn_amt_twd', 'sum'), 
    recv_count=('txn_amt_twd', 'count'), 
    recv_std=('txn_amt_twd', 'std'), 
    recv_mean=('txn_amt_twd', 'mean'), 
    recv_min=('txn_amt_twd', 'min'), 
    recv_max=('txn_amt_twd', 'max')
).reset_index().rename(columns={'to_acct': 'acct'})

# merge send and recv stats
print("merging send and recv stats...")
acct_stats = pd.merge(send_stats, recv_stats, on='acct', how='outer').fillna(0)

getting send stats...


C:\Users\0523e\AppData\Local\Temp\ipykernel_4380\1429825216.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  send_stats = df_txn.groupby('from_acct').agg(


getting recv stats...


C:\Users\0523e\AppData\Local\Temp\ipykernel_4380\1429825216.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  recv_stats = df_txn.groupby('to_acct').agg(


merging send and recv stats...


### 輸出

In [5]:
acct_stats.to_csv("feature03.csv", index=False)